# 2026 FIFA World Cup — Notebook 1: Data Collection
Data window: 2010-2022 | Sources: jfjelstul/worldcup + StatsBomb Open Data

In [1]:
import requests, warnings
import pandas as pd
import numpy as np
from datetime import datetime
import os
warnings.filterwarnings('ignore')
pd.set_option('display.float_format', '{:.4f}'.format)
DATA_DIR = '../data'
BASE_SB = 'https://raw.githubusercontent.com/statsbomb/open-data/master/data'
NAME_MAP = {
    'USA':'United States','Ivory Coast':'Cote dIvoire',
    'Korea Republic':'South Korea',
    'Bosnia-Herzegovina':'Bosnia and Herzegovina',
    'Turkiye':'Turkey','Türkiye':'Turkey',
    'IR Iran':'Iran','Congo DR':'DR Congo',
    'Czechia':'Czech Republic',
    'United States of America':'United States',
}
GROUPS = {
    'A':['Mexico','South Africa','South Korea','Czech Republic'],
    'B':['Canada','Bosnia and Herzegovina','Qatar','Switzerland'],
    'C':['Brazil','Morocco','Haiti','Scotland'],
    'D':['United States','Paraguay','Australia','Turkey'],
    'E':['Germany','Cote dIvoire','Thailand','Chile'],
    'F':['Spain','Egypt','Tunisia','Costa Rica'],
    'G':['England','Croatia','Ghana','Panama'],
    'H':['France','Senegal','Iraq','Norway'],
    'I':['Argentina','Algeria','Austria','Jordan'],
    'J':['Portugal','DR Congo','Uzbekistan','Colombia'],
    'K':['Netherlands','Iran','New Zealand','Japan'],
    'L':['Belgium','Uruguay','Sweden','Saudi Arabia'],
}
ALL_TEAMS = [t for g in GROUPS.values() for t in g]
GROUP_MAP = {t:g for g,ts in GROUPS.items() for t in ts}
print(f'Setup complete — {len(ALL_TEAMS)} teams loaded')
print(f'{datetime.now().strftime("%Y-%m-%d %H:%M")}')

Setup complete — 48 teams loaded
2026-06-12 13:06


## 1 · Load Raw Data (2010-2022)

In [2]:
matches_df  = pd.read_csv(f'{DATA_DIR}/matches.csv')
team_app_df = pd.read_csv(f'{DATA_DIR}/team_appearances.csv')
goals_df    = pd.read_csv(f'{DATA_DIR}/goals.csv')
bookings_df = pd.read_csv(f'{DATA_DIR}/bookings.csv')
for df in [matches_df,team_app_df,goals_df,bookings_df]:
    if 'match_date' in df.columns:
        df['match_date'] = pd.to_datetime(df['match_date'])
for df in [matches_df,team_app_df,goals_df,bookings_df]:
    for col in df.columns:
        if 'team' in col.lower():
            df[col] = df[col].replace(NAME_MAP)
mens_ids = matches_df[matches_df['tournament_name'].str.contains('Men',na=False)]['match_id'].unique()
matches_men = matches_df[
    matches_df['match_id'].isin(mens_ids) &
    (matches_df['match_date']>='2010-01-01') &
    (matches_df['match_date']<='2022-12-31')
].copy()
team_app_men = team_app_df[
    team_app_df['match_id'].isin(mens_ids) &
    (team_app_df['match_date']>='2010-01-01') &
    (team_app_df['match_date']<='2022-12-31')
].copy()
goals_men    = goals_df[goals_df['match_id'].isin(mens_ids)].copy()
bookings_men = bookings_df[bookings_df['match_id'].isin(mens_ids)].copy()
print(f'Matches (2010-2022): {len(matches_men)}')
print(f'Team appearances:    {len(team_app_men)}')
print(f'Goals:               {len(goals_men)}')
print(f'Bookings:            {len(bookings_men)}')
print(matches_men.groupby(matches_men['match_date'].dt.year)['match_id'].count())

Matches (2010-2022): 256
Team appearances:    512
Goals:               2720
Bookings:            2624
match_date
2010    64
2014    64
2018    64
2022    64
Name: match_id, dtype: int64


## 2 · FIFA Rankings (June 2026)

In [3]:
FIFA = {
    'Argentina':(1,1868),'France':(2,1851),'Spain':(3,1835),
    'England':(4,1792),'Brazil':(5,1781),'Belgium':(6,1763),
    'Portugal':(7,1757),'Netherlands':(8,1749),'Germany':(9,1737),
    'Colombia':(10,1720),'Morocco':(12,1700),'Mexico':(13,1689),
    'United States':(14,1680),'Switzerland':(15,1671),'Croatia':(16,1665),
    'Uruguay':(17,1658),'Japan':(18,1650),'Senegal':(19,1641),
    'South Korea':(20,1632),'Canada':(21,1625),'Austria':(22,1618),
    'Sweden':(24,1604),'Norway':(25,1596),'Chile':(26,1588),
    'Australia':(27,1580),'Turkey':(28,1572),'Czech Republic':(29,1564),
    'Saudi Arabia':(30,1556),'Ghana':(31,1548),'Cote dIvoire':(33,1532),
    'Algeria':(34,1524),'Tunisia':(35,1516),'Egypt':(36,1508),
    'Iran':(37,1500),'Costa Rica':(39,1484),'Paraguay':(40,1476),
    'Scotland':(41,1468),'South Africa':(42,1460),'Jordan':(43,1452),
    'Panama':(44,1444),'Uzbekistan':(45,1436),'New Zealand':(46,1428),
    'Qatar':(47,1420),'DR Congo':(48,1412),
    'Bosnia and Herzegovina':(49,1404),'Haiti':(50,1396),
    'Iraq':(51,1388),'Thailand':(52,1380),
}
ranks_df = pd.DataFrame(
    [{'team':t,'fifa_rank':v[0],'fifa_pts':v[1],'group':GROUP_MAP.get(t,'?')}
     for t,v in FIFA.items() if t in ALL_TEAMS]
).sort_values('fifa_rank').reset_index(drop=True)
mn,mx = ranks_df['fifa_pts'].min(),ranks_df['fifa_pts'].max()
ranks_df['rank_score'] = (ranks_df['fifa_pts']-mn)/(mx-mn)
print(f'Rankings: {len(ranks_df)} teams')
print(ranks_df.head(10)[['team','group','fifa_rank','rank_score']].to_string(index=False))

Rankings: 48 teams
       team group  fifa_rank  rank_score
  Argentina     I          1      1.0000
     France     H          2      0.9652
      Spain     F          3      0.9324
    England     G          4      0.8443
     Brazil     C          5      0.8217
    Belgium     L          6      0.7848
   Portugal     J          7      0.7725
Netherlands     K          8      0.7561
    Germany     E          9      0.7316
   Colombia     J         10      0.6967


## 3 · StatsBomb xG (2018 & 2022 only)

In [4]:
def fetch_json(url):
    r = requests.get(url, timeout=30)
    return r.json()

def get_wc_xg(season_id, label):
    print(f'  Fetching {label}...')
    matches = fetch_json(f'{BASE_SB}/matches/43/{season_id}.json')
    rows = []
    for m in matches:
        mid = m['match_id']
        home = NAME_MAP.get(m['home_team']['home_team_name'], m['home_team']['home_team_name'])
        away = NAME_MAP.get(m['away_team']['away_team_name'], m['away_team']['away_team_name'])
        try:
            events = fetch_json(f'{BASE_SB}/events/{mid}.json')
            shots = [e for e in events if e['type']['name']=='Shot']
            h_xg = sum(s['shot']['statsbomb_xg'] for s in shots if s['team']['name']==m['home_team']['home_team_name'])
            a_xg = sum(s['shot']['statsbomb_xg'] for s in shots if s['team']['name']==m['away_team']['away_team_name'])
            h_shots = sum(1 for s in shots if s['team']['name']==m['home_team']['home_team_name'])
            a_shots = sum(1 for s in shots if s['team']['name']==m['away_team']['away_team_name'])
        except:
            h_xg=a_xg=h_shots=a_shots=None
        rows.append({'match_id':mid,'date':m['match_date'],
            'stage':m['competition_stage']['name'],
            'home':home,'away':away,
            'home_score':m['home_score'],'away_score':m['away_score'],
            'home_xg':h_xg,'away_xg':a_xg,
            'home_shots':h_shots,'away_shots':a_shots,'tournament':label})
    return pd.DataFrame(rows)

print('Fetching StatsBomb xG (~60 seconds)...')
xg_2022 = get_wc_xg(106, '2022 WC')
xg_2018 = get_wc_xg(3,   '2018 WC')
xg_df = pd.concat([xg_2018,xg_2022], ignore_index=True)
xg_df.to_csv(f'{DATA_DIR}/xg_data.csv', index=False)
print(f'xG data: {len(xg_df)} matches saved')

Fetching StatsBomb xG (~60 seconds)...
  Fetching 2022 WC...
  Fetching 2018 WC...
xG data: 128 matches saved


## 4 · Compute Team Features

In [5]:
def compute_team_features(team):
    rs_row = ranks_df[ranks_df['team']==team]
    rs = rs_row['rank_score'].values[0] if len(rs_row) else 0.4
    fifa_rank = rs_row['fifa_rank'].values[0] if len(rs_row) else 50
    hist = team_app_men[team_app_men['team_name']==team]
    n = len(hist)
    if n > 0:
        wr   = hist['win'].mean()
        dr   = hist['draw'].mean()
        avgf = hist['goals_for'].mean()
        avga = hist['goals_against'].mean()
        avgd = hist['goal_differential'].mean()
    else:
        wr   = max(0.1, 0.5-(fifa_rank-1)*0.007)
        dr   = 0.25
        avgf = max(0.5, 1.8-(fifa_rank-1)*0.02)
        avga = min(2.5, 0.8+(fifa_rank-1)*0.02)
        avgd = avgf-avga
    xg_h = xg_df[xg_df['home']==team]
    xg_a = xg_df[xg_df['away']==team]
    xg_sc = list(xg_h['home_xg'].dropna())+list(xg_a['away_xg'].dropna())
    xg_cn = list(xg_h['away_xg'].dropna())+list(xg_a['home_xg'].dropna())
    avg_xg_sc = np.mean(xg_sc) if xg_sc else avgf*0.85
    avg_xg_cn = np.mean(xg_cn) if xg_cn else avga*0.85
    return {
        'team':team,'group':GROUP_MAP.get(team,'?'),
        'fifa_rank':int(fifa_rank),'rank_score':round(rs,4),
        'wc_matches':n,'win_rate':round(wr,4),'draw_rate':round(dr,4),
        'avg_gf':round(avgf,4),'avg_ga':round(avga,4),'avg_gd':round(avgd,4),
        'avg_xg_scored':round(avg_xg_sc,4),'avg_xg_conceded':round(avg_xg_cn,4),
        'xg_matches':len(xg_sc),
    }

print('Computing features for all 48 teams...')
master_df = pd.DataFrame([compute_team_features(t) for t in ALL_TEAMS]).sort_values('fifa_rank').reset_index(drop=True)
print(f'Feature matrix: {master_df.shape}')
print(master_df.head(12)[['team','group','fifa_rank','win_rate','avg_gf','avg_ga','avg_xg_scored']].to_string(index=False))

Computing features for all 48 teams...
Feature matrix: (48, 13)
       team group  fifa_rank  win_rate  avg_gf  avg_ga  avg_xg_scored
  Argentina     I          1    0.7391  1.6957  1.1739         2.4167
     France     H          2    0.6364  1.8636  0.9545         1.6967
      Spain     F          3    0.5000  1.5556  1.0000         2.3657
    England     G          4    0.4211  1.5789  1.1053         2.0444
     Brazil     C          5    0.5909  1.6364  1.0909         2.5949
    Belgium     L          6    0.7333  1.5333  0.7333         1.6419
   Portugal     J          7    0.3750  1.8125  1.2500         1.3500
Netherlands     K          8    0.7895  1.9474  0.7368         1.7823
    Germany     E          9    0.6500  2.1000  0.9000         2.4258
   Colombia     J         10    0.6667  2.0000  0.7778         1.8067
    Morocco     C         12    0.4000  0.8000  0.9000         1.2264
     Mexico     A         13    0.4000  0.9333  1.1333         1.1603


## 5 · Dark Horse Detection (rank > 25 ONLY)

In [6]:
dh = master_df[master_df['fifa_rank'] > 25].copy()
def norm(s):
    mn,mx = s.min(),s.max()
    return (s-mn)/(mx-mn) if mx>mn else pd.Series([0.5]*len(s),index=s.index)
dh['atk_score']  = norm(dh['avg_gf'])
dh['def_score']  = 1-norm(dh['avg_ga'])
dh['upset_pot']  = norm(dh['fifa_rank'])
dh['dark_horse_score'] = 0.35*dh['atk_score']+0.35*dh['def_score']+0.30*dh['upset_pot']
dark_horses = dh.sort_values('dark_horse_score',ascending=False)[
    ['team','group','fifa_rank','dark_horse_score','avg_gf','avg_ga']
].reset_index(drop=True)
print('DARK HORSE RANKINGS (rank > 25 only — powerhouses excluded)')
print('='*60)
for i,(_,row) in enumerate(dark_horses.head(8).iterrows()):
    label = 'PRIMARY' if i==0 else f'#{i+1}   '
    print(f'  {label} {row["team"]:<26} FIFA #{int(row["fifa_rank"]):<4} Score: {row["dark_horse_score"]:.3f}')

DARK HORSE RANKINGS (rank > 25 only — powerhouses excluded)
  PRIMARY Bosnia and Herzegovina     FIFA #49   Score: 0.865
  #2    Cote dIvoire               FIFA #33   Score: 0.681
  #3    New Zealand                FIFA #46   Score: 0.669
  #4    Thailand                   FIFA #52   Score: 0.654
  #5    Iraq                       FIFA #51   Score: 0.652
  #6    Haiti                      FIFA #50   Score: 0.649
  #7    DR Congo                   FIFA #48   Score: 0.645
  #8    Uzbekistan                 FIFA #45   Score: 0.637


## 6 · Player Goals Summary

In [7]:
name_col = 'player_name' if 'player_name' in goals_men.columns else [c for c in goals_men.columns if 'name' in c.lower()][0]
team_col = 'team_name' if 'team_name' in goals_men.columns else [c for c in goals_men.columns if 'team' in c.lower()][0]
player_goals = goals_men.groupby([name_col,team_col]).agg(
    total_goals=('match_id','count'),
    matches=('match_id','nunique')
).reset_index()
player_goals.columns = ['player_name','team','total_goals','matches_with_goals']
player_goals['goals_per_match'] = player_goals['total_goals']/player_goals['matches_with_goals']
player_goals = player_goals.sort_values('total_goals',ascending=False).reset_index(drop=True)
print(f'Player data: {len(player_goals)} players')
print(player_goals.head(15)[['player_name','team','total_goals','goals_per_match']].to_string(index=False))

Player data: 470 players
player_name         team  total_goals  goals_per_match
    WC-1954      Hungary           27           5.4000
    WC-1954 West Germany           25           4.1667
    WC-1958       France           23           3.8333
    WC-1950       Brazil           22           3.6667
    WC-1970       Brazil           19           3.1667
    WC-2002       Brazil           18           2.5714
    WC-2014      Germany           18           2.5714
    WC-1930    Argentina           18           3.6000
    WC-1966     Portugal           17           2.8333
    WC-1954      Austria           17           3.4000
    WC-1970 West Germany           17           2.8333
    WC-1982       France           16           2.2857
    WC-2010      Germany           16           3.2000
    WC-2022       France           16           2.6667
    WC-1974       Poland           16           2.6667


## 7 · Save All Outputs

In [8]:
master_df.to_csv(f'{DATA_DIR}/team_features.csv', index=False)
dark_horses.to_csv(f'{DATA_DIR}/dark_horse_scores.csv', index=False)
player_goals.to_csv(f'{DATA_DIR}/player_summary.csv', index=False)
print('Saved:')
print(f'  team_features.csv     {master_df.shape}')
print(f'  dark_horse_scores.csv {dark_horses.shape}')
print(f'  player_summary.csv    {player_goals.shape}')
print(f'  xg_data.csv           {xg_df.shape}')
print('Notebook 1 complete!')

Saved:
  team_features.csv     (48, 13)
  dark_horse_scores.csv (25, 6)
  player_summary.csv    (470, 5)
  xg_data.csv           (128, 12)
Notebook 1 complete!
